In [4]:
import requests
import json
import pandas as pd


In [ ]:
# pull Road Closure feed and save it to road_events_json
password = pd.read_csv("passwords.csv")
password_nps = password["password"][0]

road_events_url = "https://developer.nps.gov/api/v1/roadevents?api_key=" + password_nps

response_API = requests.get(road_events_url)
data = response_API.text
road_events_json = json.loads(data)

In [ ]:
# pull Alerts feed and save it to alerts_json
alerts_url = "https://developer.nps.gov/api/v1/alerts?limit=10000&api_key=" + password_nps

response_API = requests.get(alerts_url)
data = response_API.text
alerts_json = json.loads(data)


In [ ]:
# make a dataframe from the data sources section of Road Events
data_sources_df = pd.json_normalize(
    road_events_json, 
    record_path=['road_event_feed_info','data_sources'])

# make a dataframe from the 'core details' section of Road Events
core_details_list = [f['properties']['core_details'] for f in road_events_json['features']]
core_details_df = pd.json_normalize(core_details_list)

# merge the two dataframes on the 'data_source_id' field
road_closures_feed_df = pd.merge(
    core_details_df,                   # The main data (left)
    data_sources_df,                    # The metadata (right)
    left_on='data_source_id', 
    right_on='data_source_id',
    how='left'                     # Ensures we don't lose any road events
)

# add a column for today's date and make it the first column
today = pd.Timestamp.now().normalize()
road_closures_feed_df.insert(0, 'extraction_date', today)


# save it as a csv with today's date in the file name
today_str = today.strftime('%Y-%m-%d')
road_closures_feed_df.to_csv('Road_Closure_Feeds/' + today_str + '_Road_Closure_Feed.csv', index=False)

In [8]:
# make a dataframe out of the Alerts feed
alerts_df = pd.json_normalize(alerts_json, record_path=['data'])

# add a column for today's date and make it the first column
today = pd.Timestamp.now().normalize()
alerts_df.insert(0, 'extraction_date', today)

# save it as a csv with today's date in the file name
alerts_df.to_csv('Alerts_Feeds/' + today_str + '_Alerts_Feed.csv', index=False)